# Hücre 1 — IFDS Model Eğitimi

Bu Kaggle Notebook, IFDS codebase'i ile uyumlu iki modeli eğitir:

- **Xception CNN:** `Authentic / Tampered` binary sınıflandırma
- **EfficientNet-B4 + LSTM:** `8x8 = 64` patch skoru ve heatmap üretimi

**Dataset:** `/kaggle/input/casia-20-image-tampering-detection-dataset`  
**Çıktılar:** `/kaggle/working/xception_finetuned.h5`, `/kaggle/working/efficientnet_lstm.h5`, `/kaggle/working/training_results.json`

In [ ]:
# Hücre 2 — Kurulum
import importlib.util, os, subprocess, sys
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
required = {"tensorflow": "tensorflow", "cv2": "opencv-contrib-python", "sklearn": "scikit-learn", "tqdm": "tqdm"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
print("Eksik paketler:", missing if missing else "Yok")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Kaggle ortamındaki mevcut paketler yeterli; kurulum atlandı.")

In [ ]:
# Hücre 3 — Import ve sabitler
from __future__ import annotations
import gc, json, os, random, time
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import cv2, numpy as np, pandas as pd
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

CASIA_DIR = Path("/kaggle/input/casia-20-image-tampering-detection-dataset")
if not CASIA_DIR.exists():
    input_root = Path("/kaggle/input")
    fallback_candidates = []
    if input_root.exists():
        for candidate in input_root.iterdir():
            if candidate.is_dir() and (list(candidate.rglob("Au")) or list(candidate.rglob("Tp"))):
                fallback_candidates.append(candidate)
    if fallback_candidates:
        print("Uyarı: Verilen CASIA_DIR bulunamadı, otomatik bulunan dataset kullanılacak:", fallback_candidates[0])
        CASIA_DIR = fallback_candidates[0]
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = CASIA_DIR
XCEPTION_INPUT_SIZE = (224, 224)
LSTM_INPUT_SIZE = (256, 256)
PATCH_SIZE = (32, 32)
GRID_SIZE = 8
SEQUENCE_LENGTH = GRID_SIZE * GRID_SIZE
BATCH_SIZE = 32
EPOCHS_XCEPTION = 20
EPOCHS_LSTM = 15
LEARNING_RATE = 1e-4
FINE_TUNE_LAYERS = 20
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
RANDOM_STATE = 42
SUPPORTED_FORMATS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".tif"}
CLASS_AUTHENTIC, CLASS_TAMPERED = 0, 1
CLASS_NAMES = ["Authentic", "Tampered"]
CONFIDENCE_THRESHOLD = 0.5
XCEPTION_MODEL_PATH = OUTPUT_DIR / "xception_finetuned.h5"
LSTM_MODEL_PATH = OUTPUT_DIR / "efficientnet_lstm.h5"
XCEPTION_BEST_PATH = OUTPUT_DIR / "xception_best.h5"
LSTM_BEST_PATH = OUTPUT_DIR / "lstm_best.h5"
TRAINING_RESULTS_PATH = OUTPUT_DIR / "training_results.json"
training_results = {"xception": {}, "efficientnet_lstm": {}, "files": {}}
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)

def file_size_mb(path): return round(Path(path).stat().st_size / (1024 * 1024), 2)
def load_model_compat(path):
    try: return tf.keras.models.load_model(str(path), compile=False, safe_mode=False)
    except TypeError: return tf.keras.models.load_model(str(path), compile=False)
def plot_history(history, title):
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for k in [x for x in hist if "loss" in x]: axes[0].plot(hist[k], label=k)
    axes[0].set_title(f"{title} - Loss"); axes[0].legend()
    for k in [x for x in hist if "accuracy" in x or "auc" in x]: axes[1].plot(hist[k], label=k)
    axes[1].set_title(f"{title} - Metrics"); axes[1].legend(); plt.tight_layout(); plt.show()
def save_training_results():
    TRAINING_RESULTS_PATH.write_text(json.dumps(training_results, indent=2, ensure_ascii=False), encoding="utf-8")
    print("training_results.json yazıldı:", TRAINING_RESULTS_PATH)

print("CASIA_DIR:", CASIA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("XCEPTION_INPUT_SIZE:", XCEPTION_INPUT_SIZE, "| PATCH:", PATCH_SIZE, "| SEQUENCE_LENGTH:", SEQUENCE_LENGTH)
assert GRID_SIZE * GRID_SIZE == SEQUENCE_LENGTH == 64

In [ ]:
# Hücre 4 — GPU kontrolü
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU listesi:", gpus)
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print("Memory growth etkin:", gpu)
        except RuntimeError as exc:
            print("Memory growth ayarlanamadı:", exc)
else:
    print("GPU yok; CPU eğitimi uzun süreceği için BATCH_SIZE 16 yapılıyor.")
    BATCH_SIZE = 16
print("Aktif BATCH_SIZE:", BATCH_SIZE)

In [ ]:
# Hücre 5 — Veri yükleme ve doğrulama
def _as_path(path_value) -> Path:
    if isinstance(path_value, bytes):
        return Path(path_value.decode("utf-8"))
    if hasattr(path_value, "numpy"):
        return Path(path_value.numpy().decode("utf-8"))
    return Path(str(path_value))

def is_valid_image_file(path: str | Path) -> bool:
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

def load_image_numpy(path_value, target_size) -> np.ndarray:
    path = _as_path(path_value)
    if hasattr(target_size, "numpy"):
        target_size = tuple(int(x) for x in target_size.numpy().tolist())
    else:
        target_size = tuple(int(x) for x in target_size)
    try:
        with Image.open(path) as image:
            image = image.convert("RGB").resize((target_size[1], target_size[0]))
            array = np.asarray(image, dtype=np.float32) / 255.0
    except Exception:
        image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise ValueError(f"Görüntü okunamadı: {path}")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image_rgb = cv2.resize(image_rgb, (target_size[1], target_size[0]), interpolation=cv2.INTER_AREA)
        array = image_rgb.astype(np.float32) / 255.0
    return array.astype(np.float32)

@dataclass(frozen=True)
class DatasetSplit:
    paths: list[str]
    labels: list[int]

class ForensicDatasetBuilder:
    def __init__(self, raw_dir: str | Path = RAW_DIR) -> None:
        self.raw_dir = Path(raw_dir)
    def load_image_paths(self) -> tuple[list[str], list[int]]:
        paths, labels = [], []
        self._append_casia(paths, labels); self._append_coverage(paths, labels); self._append_columbia(paths, labels)
        return paths, labels
    def split_paths(self) -> tuple[DatasetSplit, DatasetSplit, DatasetSplit]:
        paths, labels = self.load_image_paths()
        if not paths: raise ValueError(f"Veri seti görüntüsü bulunamadı: {self.raw_dir}")
        x_temp, x_test, y_temp, y_test = train_test_split(paths, labels, test_size=1.0 - TRAIN_RATIO - VAL_RATIO, stratify=labels, random_state=RANDOM_STATE)
        val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
        x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=val_ratio_adjusted, stratify=y_temp, random_state=RANDOM_STATE)
        return DatasetSplit(list(x_train), list(y_train)), DatasetSplit(list(x_val), list(y_val)), DatasetSplit(list(x_test), list(y_test))
    def build_tf_datasets(self) -> tuple[Any, Any, Any]:
        train, val, test = self.split_paths()
        return self._create_tf_dataset(tf, train, True), self._create_tf_dataset(tf, val, False), self._create_tf_dataset(tf, test, False)
    def _create_tf_dataset(self, tf: Any, split: DatasetSplit, augment: bool) -> Any:
        ds = tf.data.Dataset.from_tensor_slices((split.paths, split.labels))
        def load_and_preprocess(path, label):
            image = tf.numpy_function(load_image_numpy, [path, XCEPTION_INPUT_SIZE], tf.float32)
            image.set_shape([XCEPTION_INPUT_SIZE[0], XCEPTION_INPUT_SIZE[1], 3])
            return image, label
        def augment_image(image, label):
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=0.12)
            image = tf.image.random_contrast(image, 0.9, 1.1)
            image = tf.image.random_saturation(image, 0.9, 1.1)
            image = tf.clip_by_value(image, 0.0, 1.0)
            return image, label
        ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        if augment:
            ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000, seed=RANDOM_STATE)
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    def _append_casia(self, paths, labels) -> None:
        au_dirs = self._find_label_dirs("Au")
        tp_dirs = self._find_label_dirs("Tp")
        print("Bulunan Au klasörleri:", [str(path) for path in au_dirs])
        print("Bulunan Tp klasörleri:", [str(path) for path in tp_dirs])
        for directory in au_dirs:
            self._append_directory(paths, labels, directory, CLASS_AUTHENTIC)
        for directory in tp_dirs:
            self._append_directory(paths, labels, directory, CLASS_TAMPERED)
    def _find_label_dirs(self, dirname: str) -> list[Path]:
        direct_candidates = [self.raw_dir / dirname, self.raw_dir / "CASIA2" / dirname]
        discovered = [path for path in direct_candidates if path.exists() and path.is_dir()]
        discovered.extend([path for path in self.raw_dir.rglob("*") if path.is_dir() and path.name.lower() == dirname.lower()])
        unique = []
        seen = set()
        for path in discovered:
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved); unique.append(path)
        return unique
    def _append_coverage(self, paths, labels) -> None:
        coverage = self.raw_dir / "Coverage"
        self._append_directory(paths, labels, coverage / "original", CLASS_AUTHENTIC); self._append_directory(paths, labels, coverage / "tampered", CLASS_TAMPERED)
    def _append_columbia(self, paths, labels) -> None:
        columbia = self.raw_dir / "Columbia"
        self._append_directory(paths, labels, columbia / "authentic", CLASS_AUTHENTIC); self._append_directory(paths, labels, columbia / "spliced", CLASS_TAMPERED)
    def _append_directory(self, paths, labels, directory: Path, label: int) -> None:
        if not directory.exists(): return
        existing = set(paths)
        for image_path in tqdm(sorted(directory.rglob("*")), desc=f"{directory} taranıyor"):
            if image_path.is_file() and image_path.suffix.lower() in SUPPORTED_FORMATS:
                path_str = str(image_path)
                if path_str not in existing and is_valid_image_file(image_path):
                    paths.append(path_str); labels.append(label); existing.add(path_str)

builder = ForensicDatasetBuilder(CASIA_DIR)
all_paths, all_labels = builder.load_image_paths()
print("Geçerli görüntü sayısı:", len(all_paths))
counts = pd.Series(all_labels).map({0: "Authentic", 1: "Tampered"}).value_counts().reindex(["Authentic", "Tampered"]).fillna(0).astype(int)
print("Toplam:", len(all_paths)); print("Authentic:", int(counts["Authentic"])); print("Tampered:", int(counts["Tampered"]))
display(counts.rename("count").to_frame())
if not CASIA_DIR.exists():
    raise FileNotFoundError(f"Dataset path yok: {CASIA_DIR}")
if len(all_paths) == 0 or set(all_labels) != {CLASS_AUTHENTIC, CLASS_TAMPERED}:
    print("İlk seviye klasörler:", [str(path) for path in CASIA_DIR.iterdir() if path.is_dir()][:30])
    print("Bulunan label seti:", sorted(set(all_labels)))
    raise ValueError("CASIA Au/Tp klasörleri bulunamadı veya iki sınıftan biri boş. Kaggle dataset'in klasör yapısını yukarıdaki çıktıyla kontrol edin.")

sample_indices = np.random.default_rng(RANDOM_STATE).choice(len(all_paths), size=min(5, len(all_paths)), replace=False)
plt.figure(figsize=(15, 4))
for plot_idx, data_idx in enumerate(sample_indices, 1):
    img = Image.open(all_paths[int(data_idx)]).convert("RGB")
    plt.subplot(1, len(sample_indices), plot_idx); plt.imshow(img); plt.title(CLASS_NAMES[all_labels[int(data_idx)]]); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Hücre 6 — TF Dataset oluşturma
train_split, val_split, test_split = builder.split_paths()
train_ds = builder._create_tf_dataset(tf, train_split, True)
val_ds = builder._create_tf_dataset(tf, val_split, False)
test_ds = builder._create_tf_dataset(tf, test_split, False)
print("Train:", len(train_split.paths), "| augmentation: açık")
print("Validation:", len(val_split.paths), "| augmentation: kapalı")
print("Test:", len(test_split.paths), "| augmentation: kapalı")
print("Batch sayıları:", {name: int(tf.data.experimental.cardinality(ds).numpy()) for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]})
batch_images, batch_labels = next(iter(train_ds))
print("Örnek batch:", batch_images.shape, batch_labels.shape)
assert batch_images.shape[1:4] == (*XCEPTION_INPUT_SIZE, 3)

In [ ]:
# Hücre 7 — Model 1: Xception build
@dataclass
class AIDetectionResult:
    model_name: str; is_forged: bool; confidence: float; class_label: str; processing_time: float
    heatmap: np.ndarray | None = None; overlay_image: np.ndarray | None = None; error_message: str | None = None

class XceptionForensicModel:
    def __init__(self, model_path: str | Path = XCEPTION_MODEL_PATH) -> None:
        self.model_path = Path(model_path); self.model: Any | None = None; self.input_size = XCEPTION_INPUT_SIZE
    def build_model(self, imagenet_weights: bool = False) -> Any:
        tf = self._tensorflow()
        from tensorflow.keras import Model, layers, optimizers, regularizers
        from tensorflow.keras.applications import Xception
        base_model = Xception(weights="imagenet" if imagenet_weights else None, include_top=False, input_shape=(*self.input_size, 3), name="xception_backbone")
        base_model.trainable = False
        inputs = tf.keras.Input(shape=(*self.input_size, 3), name="image")
        x = layers.Rescaling(2.0, offset=-1.0, name="xception_preprocess")(inputs)
        x = base_model(x, training=False)
        x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
        x = layers.BatchNormalization(name="bn_head")(x)
        x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_128")(x)
        x = layers.Dropout(0.5, name="dropout_128")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="forgery_output")(x)
        model = Model(inputs=inputs, outputs=outputs, name="xception_forensic")
        model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
        self.model = model; return model
    def load_weights(self, path: str | Path | None = None) -> None:
        model_path = Path(path) if path is not None else self.model_path
        if not model_path.exists(): raise FileNotFoundError(f"Xception model ağırlığı bulunamadı: {model_path}")
        tf = self._tensorflow()
        try: self.model = tf.keras.models.load_model(str(model_path), compile=False)
        except Exception:
            if self.model is None: self.build_model(imagenet_weights=False)
            self.model.load_weights(str(model_path))
    def unfreeze_for_finetuning(self) -> None:
        if self.model is None: raise RuntimeError("Önce build_model() çağırılmalı.")
        tf = self._tensorflow()
        from tensorflow.keras import optimizers
        base_model = self.model.get_layer("xception_backbone"); base_model.trainable = True
        for layer in base_model.layers[:-FINE_TUNE_LAYERS]: layer.trainable = False
        for layer in base_model.layers[-FINE_TUNE_LAYERS:]: layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)
        self.model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE / 10), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    def is_available(self, path: str | Path | None = None) -> bool:
        return (Path(path) if path is not None else self.model_path).exists()
    def predict(self, image: np.ndarray) -> AIDetectionResult:
        if self.model is None: raise RuntimeError("Xception modeli yüklenmedi. Önce load_weights() çağrılmalı.")
        start = time.perf_counter(); pred = float(np.asarray(self.model.predict(np.expand_dims(image.astype(np.float32), 0), verbose=0)).reshape(-1)[0])
        forged = pred >= CONFIDENCE_THRESHOLD; conf = pred if forged else 1.0 - pred
        return AIDetectionResult("Xception", bool(forged), float(conf), CLASS_NAMES[int(forged)], time.perf_counter() - start)
    @staticmethod
    def unavailable_result(message: str) -> AIDetectionResult:
        return AIDetectionResult("Xception", False, 0.0, "Unavailable", 0.0, error_message=message)
    @staticmethod
    def _tensorflow() -> Any:
        import tensorflow as tf
        return tf

xception_wrapper = XceptionForensicModel(XCEPTION_MODEL_PATH)
xception_model = xception_wrapper.build_model(imagenet_weights=True)
xception_model.summary()
xception_total_params = xception_model.count_params()
xception_trainable_params = int(np.sum([np.prod(v.shape) for v in xception_model.trainable_weights]))
print("Toplam parametre:", f"{xception_total_params:,}"); print("Eğitilebilir parametre:", f"{xception_trainable_params:,}")

In [ ]:
# Hücre 8 — Model 1: Aşama 1 eğitimi (Head)
class EpochMetricsPrinter(tf.keras.callbacks.Callback):
    def __init__(self, keys): super().__init__(); self.keys = keys
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}; print("Epoch", epoch + 1, "|", " | ".join(f"{k}={logs[k]:.4f}" for k in self.keys if k in logs))

def rebuild_xception_datasets():
    global train_ds, val_ds, test_ds
    train_ds = builder._create_tf_dataset(tf, train_split, True)
    val_ds = builder._create_tf_dataset(tf, val_split, False)
    test_ds = builder._create_tf_dataset(tf, test_split, False)
    print("Datasetler yeniden oluşturuldu. BATCH_SIZE:", BATCH_SIZE)

def fit_xception_with_auto_batch(epochs, callbacks, phase_name):
    global BATCH_SIZE
    while True:
        try:
            print(f"{phase_name}: BATCH_SIZE={BATCH_SIZE}, epochs={epochs}")
            return xception_model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, class_weight=xception_class_weight, verbose=1)
        except tf.errors.ResourceExhaustedError:
            if BATCH_SIZE <= 1: raise
            BATCH_SIZE = max(1, BATCH_SIZE // 2); print("OOM; BATCH_SIZE yarıya düşürüldü:", BATCH_SIZE)
            rebuild_xception_datasets(); gc.collect()

classes = np.array(sorted(set(train_split.labels)))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=np.array(train_split.labels))
xception_class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("class_weight:", xception_class_weight)
xception_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=4, min_delta=0.002, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(XCEPTION_BEST_PATH), monitor="val_loss", mode="min", save_best_only=True, verbose=1),
    tf.keras.callbacks.TensorBoard(log_dir=str(OUTPUT_DIR / "logs" / "xception")),
    EpochMetricsPrinter(("val_accuracy", "val_auc")),
]
history_xception_head = fit_xception_with_auto_batch(5, xception_callbacks, "Xception head eğitimi")
plot_history(history_xception_head, "Xception Head")
assert XCEPTION_BEST_PATH.exists()

In [ ]:
# Hücre 9 — Model 1: Aşama 2 fine-tune
xception_wrapper.unfreeze_for_finetuning(); xception_model = xception_wrapper.model
trainable_after = int(np.sum([np.prod(v.shape) for v in xception_model.trainable_weights]))
print("Fine-tune eğitilebilir parametre:", f"{trainable_after:,}"); assert trainable_after > xception_trainable_params
history_xception_finetune = fit_xception_with_auto_batch(EPOCHS_XCEPTION, xception_callbacks, "Xception fine-tune")
plot_history(history_xception_finetune, "Xception Fine-tune")

xception_model = load_model_compat(XCEPTION_BEST_PATH); xception_wrapper.model = xception_model
xception_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
print(dict(zip(xception_model.metrics_names, xception_model.evaluate(test_ds, verbose=1))))
val_true = np.array(val_split.labels)
val_scores = xception_model.predict(val_ds, verbose=1).reshape(-1)
threshold_grid = np.linspace(0.20, 0.80, 61)
threshold_scores = [(float(th), float(f1_score(val_true, (val_scores >= th).astype(int), zero_division=0))) for th in threshold_grid]
best_threshold, best_val_f1 = max(threshold_scores, key=lambda item: item[1])
print(f"Validation üzerinde en iyi F1 eşiği: {best_threshold:.2f} | val_f1={best_val_f1:.4f}")
y_true = np.array(test_split.labels); y_scores = xception_model.predict(test_ds, verbose=1).reshape(-1); y_pred = (y_scores >= best_threshold).astype(int)
x_metrics = {"accuracy": float(accuracy_score(y_true, y_pred)), "auc": float(roc_auc_score(y_true, y_scores)), "f1": float(f1_score(y_true, y_pred, zero_division=0)), "precision": float(precision_score(y_true, y_pred, zero_division=0)), "recall": float(recall_score(y_true, y_pred, zero_division=0)), "threshold": float(best_threshold)}
training_results["xception"].update(x_metrics); print("Xception metrics:", x_metrics)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
plt.figure(figsize=(5, 4)); plt.imshow(cm, cmap="Blues"); plt.title("Xception Confusion Matrix"); plt.xticks([0, 1], CLASS_NAMES); plt.yticks([0, 1], CLASS_NAMES)
for r in range(2):
    for c in range(2): plt.text(c, r, cm[r, c], ha="center", va="center")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.colorbar(); plt.tight_layout(); plt.show()

In [ ]:
# Hücre 10 — Model 1: Kaydet ve doğrula
xception_model.save(str(XCEPTION_MODEL_PATH)); assert XCEPTION_MODEL_PATH.exists()
training_results["files"]["xception_finetuned.h5"] = {"path": str(XCEPTION_MODEL_PATH), "size_mb": file_size_mb(XCEPTION_MODEL_PATH)}
print("Xception dosya boyutu:", training_results["files"]["xception_finetuned.h5"]["size_mb"], "MB")
reloaded_xception = load_model_compat(XCEPTION_MODEL_PATH)
sample_batch, sample_labels = next(iter(test_ds.take(1)))
sample_prediction = float(reloaded_xception.predict(sample_batch[:1], verbose=0).reshape(-1)[0])
print("Reload tek örnek tahmini:", sample_prediction, "| gerçek label:", int(sample_labels[0].numpy()))
assert 0.0 <= sample_prediction <= 1.0
print("Model başarıyla kaydedildi: xception_finetuned.h5")
save_training_results()

In [ ]:
# Hücre 11 — Model 2: EfficientNet+LSTM build
class EfficientNetLSTMModel:
    def __init__(self, model_path: str | Path = LSTM_MODEL_PATH) -> None:
        self.model_path = Path(model_path); self.model: Any | None = None; self.grid_size = GRID_SIZE; self.patch_size = PATCH_SIZE; self.sequence_length = SEQUENCE_LENGTH
    def build_model(self, imagenet_weights: bool = False) -> Any:
        tf = self._tensorflow()
        from tensorflow.keras import Model, layers, optimizers
        from tensorflow.keras.applications import EfficientNetB4
        patch_input_shape = (*self.patch_size, 3)
        weights = "imagenet" if imagenet_weights else None
        try:
            feature_extractor = EfficientNetB4(weights=weights, include_top=False, input_shape=patch_input_shape, pooling="avg", name="efficientnet_b4_features")
        except Exception as exc:
            if weights is None:
                raise
            print(f"EfficientNetB4 ImageNet ağırlıkları indirilemedi, weights=None ile devam ediliyor: {exc}")
            feature_extractor = EfficientNetB4(weights=None, include_top=False, input_shape=patch_input_shape, pooling="avg", name="efficientnet_b4_features")
        feature_extractor.trainable = False
        sequence_input = tf.keras.Input(shape=(self.sequence_length, *patch_input_shape), name="patch_sequence")
        features = layers.TimeDistributed(feature_extractor, name="patch_features")(sequence_input)
        x = layers.LSTM(256, return_sequences=True, name="lstm_256")(features); x = layers.Dropout(0.3, name="dropout_lstm_256")(x)
        x = layers.LSTM(128, return_sequences=True, name="lstm_128")(x); x = layers.Dropout(0.2, name="dropout_lstm_128")(x)
        x = layers.TimeDistributed(layers.Dense(64, activation="relu"), name="patch_dense")(x)
        patch_scores = layers.TimeDistributed(layers.Dense(1, activation="sigmoid"), name="patch_forgery_scores")(x)
        patch_scores = layers.Reshape((self.sequence_length,), name="patch_scores")(patch_scores)
        global_score = layers.Lambda(lambda values: tf.reduce_max(values, axis=-1, keepdims=True), name="global_score")(patch_scores)
        model = Model(inputs=sequence_input, outputs={"patch_scores": patch_scores, "global_score": global_score}, name="efficientnet_lstm_forensic")
        model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), loss={"patch_scores": "binary_crossentropy", "global_score": "binary_crossentropy"}, loss_weights={"patch_scores": 0.7, "global_score": 0.3}, metrics={"global_score": ["accuracy"]})
        self.model = model; return model
    def load_weights(self, path: str | Path | None = None) -> None:
        model_path = Path(path) if path is not None else self.model_path
        if not model_path.exists(): raise FileNotFoundError(f"EfficientNet+LSTM model ağırlığı bulunamadı: {model_path}")
        try: self.model = tf.keras.models.load_model(str(model_path), compile=False)
        except Exception:
            if self.model is None: self.build_model(False)
            self.model.load_weights(str(model_path))
    def is_available(self, path: str | Path | None = None) -> bool:
        return (Path(path) if path is not None else self.model_path).exists()
    def image_to_patches(self, image: np.ndarray) -> np.ndarray:
        target_height, target_width = self.grid_size * self.patch_size[0], self.grid_size * self.patch_size[1]
        resized = cv2.resize(image, (target_width, target_height), interpolation=cv2.INTER_AREA) if image.shape[:2] != (target_height, target_width) else image
        return np.asarray([resized[r*self.patch_size[0]:(r+1)*self.patch_size[0], c*self.patch_size[1]:(c+1)*self.patch_size[1], :] for r in range(self.grid_size) for c in range(self.grid_size)], dtype=np.float32)
    def predict(self, image: np.ndarray) -> AIDetectionResult:
        if self.model is None: raise RuntimeError("EfficientNet+LSTM modeli yüklenmedi. Önce load_weights() çağrılmalı.")
        start = time.perf_counter(); outputs = self.model.predict(np.expand_dims(self.image_to_patches(image), 0), verbose=0)
        patch_scores = np.asarray(outputs["patch_scores"] if isinstance(outputs, dict) else outputs[0]).reshape(-1)
        global_score = float(np.asarray(outputs["global_score"] if isinstance(outputs, dict) else outputs[1]).reshape(-1)[0])
        heatmap = patch_scores[: self.sequence_length].reshape(self.grid_size, self.grid_size); overlay = self._create_overlay(image, heatmap)
        forged = global_score >= CONFIDENCE_THRESHOLD; conf = global_score if forged else 1.0 - global_score
        return AIDetectionResult("EfficientNet + LSTM", bool(forged), float(conf), CLASS_NAMES[int(forged)], time.perf_counter() - start, heatmap=heatmap, overlay_image=overlay)
    @staticmethod
    def unavailable_result(message: str) -> AIDetectionResult:
        return AIDetectionResult("EfficientNet + LSTM", False, 0.0, "Unavailable", 0.0, error_message=message)
    def _create_overlay(self, image, heatmap):
        h, w = image.shape[:2]; heatmap_resized = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_CUBIC)
        colored = cv2.applyColorMap(np.clip(heatmap_resized * 255, 0, 255).astype(np.uint8), cv2.COLORMAP_JET)
        return cv2.addWeighted(np.clip(image * 255, 0, 255).astype(np.uint8), 0.62, cv2.cvtColor(colored, cv2.COLOR_BGR2RGB), 0.38, 0)
    @staticmethod
    def _tensorflow() -> Any:
        import tensorflow as tf
        return tf

lstm_wrapper = EfficientNetLSTMModel(LSTM_MODEL_PATH)
lstm_model = lstm_wrapper.build_model(imagenet_weights=True)
lstm_model.summary()
assert lstm_wrapper.sequence_length == 64 and lstm_wrapper.grid_size == 8

In [ ]:
# Hücre 12 — Model 2: Patch dataset hazırlama
sample_np = np.asarray(Image.open(test_split.paths[0]).convert("RGB").resize(LSTM_INPUT_SIZE)).astype(np.float32) / 255.0
sample_patches = lstm_wrapper.image_to_patches(sample_np)
print("Patch shape:", sample_patches.shape); assert sample_patches.shape == (64, 32, 32, 3)
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flat): ax.imshow(sample_patches[i]); ax.axis("off")
plt.suptitle("8x8 patch grid"); plt.tight_layout(); plt.show()

LSTM_BATCH_SIZE = BATCH_SIZE
def make_lstm_dataset(split, augment, batch_size):
    ds = tf.data.Dataset.from_tensor_slices((split.paths, split.labels))
    def load_to_patches(path, label):
        image = tf.numpy_function(load_image_numpy, [path, LSTM_INPUT_SIZE], tf.float32)
        image.set_shape([LSTM_INPUT_SIZE[0], LSTM_INPUT_SIZE[1], 3])
        if augment:
            image = tf.image.random_flip_left_right(image); image = tf.image.random_brightness(image, 0.2); image = tf.image.random_contrast(image, 0.8, 1.2)
        patches = tf.image.extract_patches(tf.expand_dims(image, 0), sizes=[1, 32, 32, 1], strides=[1, 32, 32, 1], rates=[1, 1, 1, 1], padding="VALID")
        patches = tf.reshape(patches, (SEQUENCE_LENGTH, 32, 32, 3))
        label = tf.cast(label, tf.float32)
        return patches, {"patch_scores": tf.repeat(label, SEQUENCE_LENGTH), "global_score": tf.reshape(label, (1,))}
    ds = ds.map(load_to_patches, num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds = ds.shuffle(1000, seed=RANDOM_STATE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
def rebuild_lstm_datasets():
    global lstm_train_ds, lstm_val_ds, lstm_test_ds
    lstm_train_ds = make_lstm_dataset(train_split, True, LSTM_BATCH_SIZE)
    lstm_val_ds = make_lstm_dataset(val_split, False, LSTM_BATCH_SIZE)
    lstm_test_ds = make_lstm_dataset(test_split, False, LSTM_BATCH_SIZE)
    print("LSTM datasetler hazır. LSTM_BATCH_SIZE:", LSTM_BATCH_SIZE)
rebuild_lstm_datasets()
patch_batch, target_batch = next(iter(lstm_train_ds))
print(patch_batch.shape, target_batch["patch_scores"].shape, target_batch["global_score"].shape)
assert patch_batch.shape[1:] == (64, 32, 32, 3)

In [ ]:
# Hücre 13 — Model 2: Eğitim
class LSTMEpochPrinter(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}; print(f"Epoch {epoch+1}: loss={logs.get('loss', 0):.4f} | global_score_accuracy={logs.get('global_score_accuracy', 0):.4f} | val_global_score_accuracy={logs.get('val_global_score_accuracy', 0):.4f}")
def make_lstm_callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor="val_global_score_accuracy", mode="max", patience=5, restore_best_weights=True), tf.keras.callbacks.ModelCheckpoint(str(LSTM_BEST_PATH), monitor="val_global_score_accuracy", mode="max", save_best_only=True, verbose=1), LSTMEpochPrinter()]
def rebuild_lstm_model():
    global lstm_wrapper, lstm_model
    tf.keras.backend.clear_session(); gc.collect()
    lstm_wrapper = EfficientNetLSTMModel(LSTM_MODEL_PATH); lstm_model = lstm_wrapper.build_model(imagenet_weights=True)
def fit_lstm_with_auto_batch():
    global LSTM_BATCH_SIZE, lstm_callbacks
    while True:
        try:
            print("EfficientNet+LSTM eğitimi. LSTM_BATCH_SIZE:", LSTM_BATCH_SIZE)
            return lstm_model.fit(lstm_train_ds, validation_data=lstm_val_ds, epochs=EPOCHS_LSTM, callbacks=lstm_callbacks, verbose=1)
        except tf.errors.ResourceExhaustedError:
            if LSTM_BATCH_SIZE <= 1: raise
            LSTM_BATCH_SIZE = max(1, LSTM_BATCH_SIZE // 2); print("OOM; LSTM_BATCH_SIZE yarıya düştü:", LSTM_BATCH_SIZE)
            rebuild_lstm_datasets(); rebuild_lstm_model(); lstm_callbacks = make_lstm_callbacks()
lstm_callbacks = make_lstm_callbacks()
history_lstm = fit_lstm_with_auto_batch()
plot_history(history_lstm, "EfficientNet+LSTM")
assert LSTM_BEST_PATH.exists()
lstm_model = load_model_compat(LSTM_BEST_PATH); lstm_wrapper.model = lstm_model
outputs = lstm_model.predict(lstm_test_ds, verbose=1)
scores = np.asarray(outputs["global_score"] if isinstance(outputs, dict) else outputs[1]).reshape(-1)
y_true = np.array(test_split.labels); y_pred = (scores >= CONFIDENCE_THRESHOLD).astype(int)
l_metrics = {"accuracy": float(accuracy_score(y_true, y_pred)), "auc": float(roc_auc_score(y_true, scores)), "f1": float(f1_score(y_true, y_pred, zero_division=0)), "precision": float(precision_score(y_true, y_pred, zero_division=0)), "recall": float(recall_score(y_true, y_pred, zero_division=0)), "batch_size": int(LSTM_BATCH_SIZE), "labeling_note": "Weak patch labels: each patch inherits image-level CASIA label."}
training_results["efficientnet_lstm"].update(l_metrics); print("EfficientNet+LSTM metrics:", l_metrics)

In [ ]:
# Hücre 14 — Model 2: Kaydet ve doğrula
lstm_model.save(str(LSTM_MODEL_PATH)); assert LSTM_MODEL_PATH.exists()
training_results["files"]["efficientnet_lstm.h5"] = {"path": str(LSTM_MODEL_PATH), "size_mb": file_size_mb(LSTM_MODEL_PATH)}
training_results["files"]["xception_finetuned.h5"] = {"path": str(XCEPTION_MODEL_PATH), "size_mb": file_size_mb(XCEPTION_MODEL_PATH)}
print("EfficientNet+LSTM dosya boyutu:", training_results["files"]["efficientnet_lstm.h5"]["size_mb"], "MB")
validation_wrapper = EfficientNetLSTMModel(LSTM_MODEL_PATH); validation_wrapper.model = load_model_compat(LSTM_MODEL_PATH)
lstm_result = validation_wrapper.predict(sample_np.astype(np.float32))
print("Reload class_label:", lstm_result.class_label, "| confidence:", lstm_result.confidence)
assert lstm_result.heatmap is not None and lstm_result.heatmap.shape == (8, 8) and lstm_result.overlay_image is not None
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(sample_np); axes[0].set_title("Original"); axes[0].axis("off")
im = axes[1].imshow(lstm_result.heatmap, cmap="jet", vmin=0, vmax=1); axes[1].set_title("8x8 Heatmap"); axes[1].axis("off"); plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(lstm_result.overlay_image); axes[2].set_title("Overlay"); axes[2].axis("off"); plt.tight_layout(); plt.show()
print("Model başarıyla kaydedildi: efficientnet_lstm.h5")
save_training_results()
display(pd.DataFrame([{"model": "Xception", **training_results["xception"]}, {"model": "EfficientNet+LSTM", **{k: v for k, v in training_results["efficientnet_lstm"].items() if k != "labeling_note"}}]))
display(pd.DataFrame([{"file": name, **info} for name, info in training_results["files"].items()]))

# Hücre 15 — Özet ve indirme talimatları

Son kod hücresi iki modelin test skorlarını tablo halinde gösterir, dosya boyutlarını listeler ve `/kaggle/working/training_results.json` dosyasına yazar.

## Kaggle Output sekmesinden indirilecek dosyalar

- `/kaggle/working/xception_finetuned.h5`
- `/kaggle/working/efficientnet_lstm.h5`
- `/kaggle/working/training_results.json`

## Codebase'e yerleştirme

İndirdiğiniz dosyaları projedeki `data/models/` klasörüne koyun:

- `xception_finetuned.h5` → `data/models/xception_finetuned.h5`
- `efficientnet_lstm.h5` → `data/models/efficientnet_lstm.h5`

Bu işlemden sonra IFDS Streamlit uygulaması ek ayar gerektirmeden AI inference için bu ağırlıkları kullanır.